## 1. Read the prepared datasets

We load the prepared energy and weather files from the processed data folder.

Energy contains load, solar generation and residual load (`residual_load = load - solar`). Weather contains the mean temperature and shortwave radiation across the selected locations.

The paths work when the notebook is run from the project root or the notebooks folder.

In [5]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

input_dir = root / "data" / "processed data"
output_file = root / "data" / "merged data" / "energy-weather_2025.csv"

energy = pd.read_csv(input_dir / "energy-data_2025.csv",
                     usecols=["timestamp", "load", "solar", "residual_load"])
weather = pd.read_csv(input_dir / "weather-data_2025.csv",
                      usecols=["timestamp", "temperature_2m", "shortwave_radiation"])

print(f"Energy rows: {len(energy):,}")
print(f"Weather rows: {len(weather):,}")
display(energy.head(3), weather.head(3))


Energy rows: 8,760
Weather rows: 8,760


,timestamp,load,solar,residual_load
0,2025-01-01T00:00:00+01:00,7880.5,0.0,7880.5
1,2025-01-01T01:00:00+01:00,8432.4,0.0,8432.4
2,2025-01-01T02:00:00+01:00,8373.9,0.0,8373.9


,timestamp,temperature_2m,shortwave_radiation
0,2025-01-01T00:00:00+01:00,-1.4571,0.0
1,2025-01-01T01:00:00+01:00,-1.7857,0.0
2,2025-01-01T02:00:00+01:00,-2.4000,0.0


## 2. Check coverage and measurements before merging

Both prepared datasets already use the same ISO 8601 timestamp strings with explicit Swiss winter or summer offsets.

For each dataset, we expect all 8,760 hourly instants of the local year 2025 in Europe/Zurich. 

We format the reference calendar in the same way as the input files and check for missing hours, unexpected timestamps, duplicates and missing timestamp values.

We also check that the measurements are numeric and complete. 

Any non-zero count stops execution before merging or exporting.

In [6]:
expected = pd.date_range("2025-01-01", "2026-01-01", freq="h",
                         tz="Europe/Zurich", inclusive="left")
expected = pd.Index([t.isoformat(timespec="seconds") for t in expected],
                    name="timestamp")

checks = {}
for name, frame in {"Energy": energy, "Weather": weather}.items():
    actual = pd.Index(frame["timestamp"])
    measurements = frame.columns.drop("timestamp")
    for column in measurements:
        frame[column] = pd.to_numeric(frame[column], errors="raise")

    checks[name] = {
        "Missing expected hours": len(expected.difference(actual)),
        "Unexpected timestamps": len(actual.difference(expected)),
        "Duplicate timestamps": int(actual.duplicated().sum()),
        "Missing timestamps": int(frame["timestamp"].isna().sum()),
        "Missing measurement values": int(frame[measurements].isna().sum().sum()),
    }

checks = pd.DataFrame(checks)
display(checks)
if checks.to_numpy().any():
    raise ValueError("Review the non-zero counts before merging.")


,Energy,Weather
Missing expected hours,0,0
Unexpected timestamps,0,0
Duplicate timestamps,0,0
Missing timestamps,0,0
Missing measurement values,0,0


## 3. Merge energy and weather by timestamp

We use energy as the base and add the weather measurements with a left join directly on the unchanged timestamp strings.

The `one_to_one` validation requires an unambiguous match between the datasets. The merge indicator confirms that every energy row has a weather match.

We arrange the result in the order of the reference calendar. This preserves chronological order even during the autumn clock change, when local hours repeat with different offsets.

We stop if the merge changes the expected row count or leaves missing values.

In [7]:
merged = energy.merge(weather, on="timestamp", how="left",
                      validate="one_to_one", indicator=True)

if not merged["_merge"].eq("both").all():
    raise ValueError("Some energy timestamps have no matching weather data.")

merged = (
    merged.drop(columns="_merge")
    .set_index("timestamp")
    .reindex(expected)
    .reset_index()
)

if len(merged) != len(expected) or merged.isna().any().any():
    raise ValueError("The merged data must contain 8,760 complete hourly records.")

print(f"Merged rows: {len(merged):,}")
display(merged.head(3))


Merged rows: 8,760


,timestamp,load,solar,residual_load,temperature_2m,shortwave_radiation
0,2025-01-01T00:00:00+01:00,7880.5,0.0,7880.5,-1.4571,0.0
1,2025-01-01T01:00:00+01:00,8432.4,0.0,8432.4,-1.7857,0.0
2,2025-01-01T02:00:00+01:00,8373.9,0.0,8373.9,-2.4000,0.0


## 4. Export the merged data

We save the six columns, including `residual_load`, to `data/merged data/energy-weather_2025.csv` without the dataframe index.

The timestamp strings retain exactly the format supplied by the prepared datasets. No timezone conversion or timestamp reformatting is needed. The two input files remain unchanged.

In [8]:
output_file.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(output_file, index=False)

print(f"Saved {len(merged):,} hourly records to {output_file}")
display(merged.head(3), merged.tail(3))


Saved 8,760 hourly records to /home/simon/dspro01_hs26_group-XX-/data/merged data/energy-weather_2025.csv


,timestamp,load,solar,residual_load,temperature_2m,shortwave_radiation
0,2025-01-01T00:00:00+01:00,7880.5,0.0,7880.5,-1.4571,0.0
1,2025-01-01T01:00:00+01:00,8432.4,0.0,8432.4,-1.7857,0.0
2,2025-01-01T02:00:00+01:00,8373.9,0.0,8373.9,-2.4000,0.0


,timestamp,load,solar,residual_load,temperature_2m,shortwave_radiation
8757,2025-12-31T21:00:00+01:00,8201.5,0.0,8201.5,-2.2857,0.0
8758,2025-12-31T22:00:00+01:00,8209.3,0.0,8209.3,-2.6571,0.0
8759,2025-12-31T23:00:00+01:00,8305.2,0.0,8305.2,-2.9000,0.0
